# Data Cleaning & Preparation

This notebook creates cleaned versions of **Customers, Accounts, Transactions and Branches** for SQL analysis and Power BI reporting.

### Cleaning Performed
- Remove exact duplicate rows.
- Trim whitespace from text fields.
- Standardize date fields.
- Remove duplicate primary-key records using a documented **keep-first** rule.
- Remove invalid non-null foreign-key references; missing foreign keys are retained for review.
- Do not automatically delete negative account balances because they may represent legitimate banking conditions.
- Save cleaned datasets under `cleaned_data`.

In [10]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path('Raw_Data')
OUT_DIR = Path('Cleaned_Data')
OUT_DIR.mkdir(exist_ok=True)

customers = pd.read_csv(DATA_DIR/'customers.csv')
accounts = pd.read_csv(DATA_DIR/'accounts.csv')
transactions = pd.read_csv(DATA_DIR/'transactions.csv')
branches = pd.read_csv(DATA_DIR/'branches.csv')

## 1. Remove exact duplicate rows

In [11]:
datasets = {
    'Customers': customers,
    'Accounts': accounts,
    'Transactions': transactions,
    'Branches': branches
}

for name, df in datasets.items():
    before = len(df)
    datasets[name] = df.drop_duplicates().copy()
    print(f'{name}: removed {before - len(datasets[name]):,} exact duplicate rows')

customers = datasets['Customers']
accounts = datasets['Accounts']
transactions = datasets['Transactions']
branches = datasets['Branches']

Customers: removed 11 exact duplicate rows
Accounts: removed 16 exact duplicate rows
Transactions: removed 500 exact duplicate rows
Branches: removed 0 exact duplicate rows


## 2. Clean text fields

In [12]:
for df in [customers, accounts, transactions, branches]:
    text_cols = df.select_dtypes(include=['object', 'string']).columns
    for col in text_cols:
        df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)

print('Text fields trimmed.')

Text fields trimmed.


## 3. Standardize date columns

In [13]:
date_columns = {
    'Customers': (customers, ['DateOfBirth']),
    'Accounts': (accounts, ['OpeningDate']),
    'Transactions': (transactions, ['TransactionDate'])
}

for name, (df, cols) in date_columns.items():
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce').dt.strftime('%Y-%m-%d')

print('Date fields standardized to YYYY-MM-DD.')

Date fields standardized to YYYY-MM-DD.


## 4. Remove duplicate primary-key records

After exact duplicates are removed, any remaining duplicate primary-key values are handled using a documented **keep-first** rule.

In [14]:
pk_map = {
    'CustomerID': customers,
    'AccountID': accounts,
    'TransactionID': transactions,
    'BranchID': branches
}

for pk, df in pk_map.items():
    if pk in df.columns:
        before = len(df)
        duplicate_rows = int(df[pk].duplicated(keep=False).sum())
        df.drop_duplicates(subset=[pk], keep='first', inplace=True)
        print(f'{pk}: {duplicate_rows:,} rows involved in duplicate IDs; {before - len(df):,} rows removed')

CustomerID: 0 rows involved in duplicate IDs; 0 rows removed
AccountID: 0 rows involved in duplicate IDs; 0 rows removed
TransactionID: 0 rows involved in duplicate IDs; 0 rows removed
BranchID: 0 rows involved in duplicate IDs; 0 rows removed


## 5. Validate foreign-key relationships

In [15]:
# Missing foreign keys are retained for review; invalid non-null references are removed.
# Only relationships supported by the actual schema are checked.
# Accounts does NOT contain BranchID, so no Accounts -> Branches check is performed.

fk_rules = [
    (accounts, 'CustomerID', customers, 'CustomerID'),
    (transactions, 'AccountOriginID', accounts, 'AccountID'),
    (transactions, 'AccountDestinationID', accounts, 'AccountID'),
    (transactions, 'BranchID', branches, 'BranchID')
]

for child_df, child_col, parent_df, parent_col in fk_rules:
    if child_col in child_df.columns and parent_col in parent_df.columns:
        valid_parent_ids = set(parent_df[parent_col].dropna())
        invalid_mask = child_df[child_col].notna() & ~child_df[child_col].isin(valid_parent_ids)
        invalid_count = int(invalid_mask.sum())
        child_df.drop(index=child_df.index[invalid_mask], inplace=True)
        print(f'{child_col} -> {parent_col}: removed {invalid_count:,} invalid references')

CustomerID -> CustomerID: removed 0 invalid references
AccountOriginID -> AccountID: removed 0 invalid references
AccountDestinationID -> AccountID: removed 0 invalid references
BranchID -> BranchID: removed 0 invalid references


## 6. Final quality review before saving

In [16]:
cleaned = {
    'Customers': customers,
    'Accounts': accounts,
    'Transactions': transactions,
    'Branches': branches
}

review_rows = []
for name, df in cleaned.items():
    review_rows.append({
        'Dataset': name,
        'Rows': len(df),
        'Columns': len(df.columns),
        'Missing_Cells': int(df.isna().sum().sum()),
        'Exact_Duplicates': int(df.duplicated().sum())
    })

review = pd.DataFrame(review_rows)
display(review)

,Dataset,Rows,Columns,Missing_Cells,Exact_Duplicates
0,Customers,1100,6,76,0
1,Accounts,1651,6,33,0
2,Transactions,49500,8,990,0
3,Branches,50,3,0,0


## 7. Save cleaned datasets

In [17]:
cleaned = {
    'Customers': customers,
    'Accounts': accounts,
    'Transactions': transactions,
    'Branches': branches
}

for name, df in cleaned.items():
    df.to_csv(OUT_DIR/f'{name}_cleaned.csv', index=False)
    print(f'{name}: {len(df):,} rows saved')


Customers: 1,100 rows saved
Accounts: 1,651 rows saved
Transactions: 49,500 rows saved
Branches: 50 rows saved
